# ETL – Silver → Gold (Star Schema)

Este notebook implementa o processo de ETL responsável por transformar os dados da camada **Silver**
em um modelo **dimensional (Star Schema)** na camada **Gold** do Data Warehouse.

A camada Gold tem como objetivo suportar análises analíticas e consultas gerenciais,
organizando os dados em **tabelas dimensão** e uma **tabela fato**.


In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

# Configurações do Banco
DB_HOST = 'localhost'
DB_PORT = '5432'
DB_NAME = 'ceap_dw'
DB_USER = 'admin'
DB_PASS = 'admin_password'

db_url = f"postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(db_url)

print("1. Conexão com o banco estabelecida.")


1. Conexão com o banco estabelecida.


## EXTRACT

Nesta etapa são extraídos os dados já tratados e padronizados da tabela
`s‍ilver.tb_reembolso`, que serve como base para a construção da camada Gold.


In [ ]:
df_silver = pd.read_sql_query(
    "SELECT * FROM silver.tb_reembolso",
    engine
)

print(f"2. Dados extraídos da Silver: {len(df_silver)} registros.")


## TRANSFORM

Nesta etapa os dados da camada Silver são reorganizados para o modelo dimensional,
sendo criados DataFrames correspondentes às **tabelas dimensão** e à **tabela fato**.


In [ ]:
df_dim_tmp = (
    df_silver[['receipt_date']]
    .dropna()
    .drop_duplicates()
    .rename(columns={'receipt_date': 'dat_cmp'})
)

df_dim_tmp['num_ano'] = pd.to_datetime(df_dim_tmp['dat_cmp']).dt.year
df_dim_tmp['num_mes'] = pd.to_datetime(df_dim_tmp['dat_cmp']).dt.month
df_dim_tmp['num_dia'] = pd.to_datetime(df_dim_tmp['dat_cmp']).dt.day
df_dim_tmp['num_tri'] = pd.to_datetime(df_dim_tmp['dat_cmp']).dt.quarter
df_dim_tmp['num_sem'] = pd.to_datetime(df_dim_tmp['dat_cmp']).dt.isocalendar().week

print(f"   - DIM_TMP criada: {len(df_dim_tmp)} registros.")

In [ ]:
df_dim_dpt = (
    df_silver[['deputy_id', 'deputy_name', 'state_code',
               'political_party', 'party_ideology', 'party_regdate']]
    .drop_duplicates()
    .rename(columns={
        'deputy_id': 'cod_dpt',
        'deputy_name': 'nom_par',
        'state_code': 'sgl_est',
        'political_party': 'sgl_prt',
        'party_ideology': 'txt_ide',
        'party_regdate': 'dat_cri'
    })
)

print(f"   - DIM_DPT criada: {len(df_dim_dpt)} registros.")


In [ ]:
df_dim_frn = (
    df_silver[['receipt_social_security_number', 'establishment_name']]
    .drop_duplicates()
    .rename(columns={
        'receipt_social_security_number': 'cod_doc',
        'establishment_name': 'nom_frn'
    })
)

print(f"   - DIM_FRN criada: {len(df_dim_frn)} registros.")


In [ ]:
df_dim_cat = (
    df_silver[['receipt_description']]
    .dropna()
    .drop_duplicates()
    .rename(columns={'receipt_description': 'nom_cat_ori'})
)

df_dim_cat['nom_cat_pad'] = df_dim_cat['nom_cat_ori'].str.upper().str.strip()

print(f"   - DIM_CAT criada: {len(df_dim_cat)} registros.")


## LOAD

Nesta etapa são criadas as tabelas da camada Gold no banco de dados
e realizadas as cargas das dimensões e da tabela fato.


In [ ]:
with engine.connect() as conn:
    conn.execute(text("""
        CREATE SCHEMA IF NOT EXISTS gold;
        TRUNCATE TABLE gold.fat_rmb CASCADE;
    """))
    conn.commit()

print("4. Estrutura da camada Gold preparada.")


In [ ]:
df_dim_dpt.to_sql('dim_dpt', engine, schema='gold', if_exists='append', index=False)
df_dim_cat.to_sql('dim_cat', engine, schema='gold', if_exists='append', index=False)
df_dim_frn.to_sql('dim_frn', engine, schema='gold', if_exists='append', index=False)
df_dim_tmp.to_sql('dim_tmp', engine, schema='gold', if_exists='append', index=False)

print("   - Dimensões carregadas com sucesso.")


## Construção da Tabela Fato

A tabela fato é construída a partir do relacionamento entre a camada Silver
e as tabelas dimensão da camada Gold, utilizando chaves substitutas.
